# Databricks backend clustering job
This notebook uses a configurable backend URL to check service health, verify ML endpoints, and trigger clustering. Set `backend_base_url` as a Databricks widget or environment variable when configuring the job.

In [ ]:
import os
import json
from datetime import datetime, timezone

try:
    from pyspark.dbutils import DBUtils  # Databricks runtime
except Exception:
    DBUtils = None

def _widget(name: str, default: str) -> str:
    try:
        value = dbutils.widgets.get(name)  # type: ignore[name-defined]
        return value or default
    except Exception:
        return os.getenv(name.upper(), default)

BACKEND_BASE_URL = _widget("backend_base_url", "http://127.0.0.1:8000").rstrip("/")
REQUEST_TIMEOUT_SECONDS = int(_widget("request_timeout_seconds", "120"))
CLUSTER_LIMIT = int(_widget("cluster_limit", "500"))
N_CLUSTERS = int(_widget("n_clusters", "8"))

print({
    "backend_base_url": BACKEND_BASE_URL,
    "request_timeout_seconds": REQUEST_TIMEOUT_SECONDS,
    "cluster_limit": CLUSTER_LIMIT,
    "n_clusters": N_CLUSTERS,
    "runtime_utc": datetime.now(timezone.utc).isoformat(),
})

In [ ]:
import requests

session = requests.Session()

def request_json(method: str, path: str, **kwargs):
    url = f"{BACKEND_BASE_URL}{path}"
    response = session.request(method, url, timeout=REQUEST_TIMEOUT_SECONDS, **kwargs)
    response.raise_for_status()
    try:
        return response.json()
    except ValueError:
        return {"text": response.text}

health = request_json("get", "/api/v1/health")
ml_health = request_json("get", "/api/v1/ml/health")
ml_status = request_json("get", "/api/v1/ml/status")

print("Backend health:")
print(json.dumps(health, indent=2, default=str))

print("\nML health:")
print(json.dumps(ml_health, indent=2, default=str))

print("\nML status:")
print(json.dumps(ml_status, indent=2, default=str))

cluster_result = request_json(
    "post",
    "/api/v1/ml/cluster/run",
    params={"limit": CLUSTER_LIMIT, "n_clusters": N_CLUSTERS},
 )

print("\nCluster result:")
print(json.dumps(cluster_result, indent=2, default=str))

summary = {
    "backend_base_url": BACKEND_BASE_URL,
    "health_status": health.get("status"),
    "classifier_loaded": ml_health.get("classifier_loaded"),
    "classifier_model": ml_health.get("classifier_model"),
    "clustered": cluster_result.get("clustered"),
    "embeddings_stored": cluster_result.get("embeddings_stored"),
}

print("\nRun summary:")
print(json.dumps(summary, indent=2, default=str))